In [12]:
#TRIAGE 
import cv2
import numpy as np 

img=cv2.imread('./task_images/1.png')
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
# (HSV space separates color information from image hence making colour separation possible)


lower_blue = np.array([90, 50, 50])
upper_blue = np.array([140, 255, 255])
# these ranges detect blue pixels in the image 
ocean_mask = cv2.inRange(hsv, lower_blue, upper_blue)

# Creates a binary mask
# White → ocean
# Black → non-ocean


land_mask = cv2.bitwise_not(ocean_mask)
# Land is the complement of the ocean mask

# making a copy so that original image remains unchanged
segmented = img.copy()
segmented[ocean_mask > 0] = [255, 0, 0] 
segmented[land_mask > 0] = [0, 255, 0]


# Grayscale simplifies processing
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


# Binary inversion helps in highlights geometric symbols against a bright background
_, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY_INV)


# A list of contours for each one object
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)


camps = []

# Detecting circular rescue camps
circles = cv2.HoughCircles(
    gray,
    cv2.HOUGH_GRADIENT,
    # Resolution ratio
    dp=1.2,
    # Minimum distance between two detected circles (Prevents duplicate detection)
    minDist=50,
    param1=100,
    param2=30,
    # Ensures only camp-sized circles are detected
    minRadius=20,
    maxRadius=60
)

# Mapping
camp_meta = [
    ("blue", 4),
    ("pink", 3),
    ("grey", 2)
]

if circles is not None:
    circles = np.uint16(np.around(circles))
    for i, circle in enumerate(circles[0]):
        # (x, y) is center of the rescue camp
        x, y, r = circle
        name, capacity = camp_meta[i]
        # Each camp stored as a dictionary
        camps.append({
            "name": name,
            "pos": (x, y),
            "capacity": capacity
        })


casualties = []
casualty_id = 0

for cnt in contours:
    area = cv2.contourArea(cnt)
    # filtering contours by area to remove noise
    if area < 200:
        continue  
    # Used to compute centroid
    M = cv2.moments(cnt)
    if M["m00"] == 0:
        continue
    # Centroids are computed using image moments
    cx = int(M["m10"] / M["m00"])
    cy = int(M["m01"] / M["m00"])

    epsilon = 0.04 * cv2.arcLength(cnt, True)
    # Simplifies contour into vertices
    approx = cv2.approxPolyDP(cnt, epsilon, True)

    # Vertices from polygon approximation are used for shape classification
    if len(approx) == 3:
        age_score = 2    # triangle = elderly
    elif len(approx) == 4:
        age_score = 1    # square = adult
    else:
        age_score = 3    # star = child

    # Emergency severity 
    emergency_score = 2

    casualties.append({
        "id": casualty_id,
        "pos": (cx, cy),
        "priority": age_score * emergency_score
    })

    casualty_id += 1


import math

# distance between two points in cartesian plane

def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)


# I maintained a mapping from each rescue camp to the list of casualties assigned to it 
assignments = {
    "blue": [],
    "pink": [],
    "grey": []
}
# I used a set to ensure each casualty is assigned only once


assigned = set()


# This will store all possible (casualty,camp) combinations with their score.
pairs = []


# I evaluated every casualty–camp pair to compute a comparative score
for c in casualties:
    for camp in camps:
        dist = distance(c["pos"], camp["pos"])

# The score balances emergency and reachability between casualty and camp Priority dominates but distance acts as a secondary factor;

        score = 5 * c["priority"] - dist
        pairs.append((score, c["id"], camp["name"]))


# I sorted all possible assignments by descending score and then assign accordinggly


pairs.sort(reverse=True)


for score, cid, camp_name in pairs:
    # Skip the iteration if casualty is already assigned
    if cid in assigned:
        continue
    # I check camp capacity during assignment
    if len(assignments[camp_name]) >= next(c["capacity"] for c in camps if c["name"] == camp_name):
        continue

    # Casualty is assigned
    assignments[camp_name].append(cid)
    assigned.add(cid)

# Dictionary to store total priority per camp
camp_priority = {}

# I compute total priority per camp by summing individual casualty priorities

for camp_name, cas_ids in assignments.items():
    camp_priority[camp_name] = sum(
        c["priority"] for c in casualties if c["id"] in cas_ids
    )

print("Camp priorities:", camp_priority)


total_priority = sum(camp_priority.values())
total_casualties = len(casualties)

Pr = total_priority / total_casualties
print("Rescue Ratio (Pr):", Pr)





Camp priorities: {'blue': 2, 'pink': 0, 'grey': 0}
Rescue Ratio (Pr): 2.0


C:\Users\vaats\AppData\Local\Temp\ipykernel_17880\2382869345.py:124: RuntimeWarning: overflow encountered in scalar subtract
  return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)


In [ ]:

# TRIAGE SYSTEM 

import cv2
import numpy as np
import math

# loading Image and coverting color

img = cv2.imread("./task_images/1.png")
assert img is not None, "Image not found!"

hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# camp detection using color segmentation


camps = []

# HSV color ranges for camps
pink_mask = cv2.inRange(hsv, np.array([140, 50, 50]), np.array([170, 255, 255]))
grey_mask = cv2.inRange(hsv, np.array([0, 0, 180]), np.array([180, 40, 255]))

camp_masks = {
    "pink": (pink_mask, 3),
    "grey": (grey_mask, 2)
}

for name, (mask, capacity) in camp_masks.items():
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        if cv2.contourArea(cnt) < 500:
            continue

        M = cv2.moments(cnt)
        if M["m00"] == 0:
            continue

        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])

        camps.append({
            "name": name,
            "pos": (cx, cy),
            "capacity": capacity
        })

# removing duplicate camps

unique_camps = {}
for camp in camps:
    name = camp["name"]
    if name not in unique_camps:
        unique_camps[name] = camp
    else:
        # keep the one farther from image border 
        old = unique_camps[name]
        if camp["pos"][0] > old["pos"][0]:
            unique_camps[name] = camp

camps = list(unique_camps.values())


print("Detected camps:", camps)




# defining the distance function


def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)


# main triage assessment logic  


assignments = {c["name"]: [] for c in camps}
assigned = set()

for casualty in sorted(casualties, key=lambda x: -x["priority"]):
    cid = casualty["id"]

    best_camp = None
    best_dist = float("inf")

    for camp in camps:
        cname = camp["name"]
        if len(assignments[cname]) >= camp["capacity"]:
            continue

        d = distance(casualty["pos"], camp["pos"])
        if d < best_dist:
            best_dist = d
            best_camp = cname

    if best_camp is not None:
        assignments[best_camp].append(cid)
        assigned.add(cid)

# casualty detection

casualties = []
casualty_id = 0

# Adaptive threshold (handles lighting + grey symbols)
binary = cv2.adaptiveThreshold(
    gray,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV,
    11,
    2
)

# Light noise removal (NOT aggressive)
kernel = np.ones((3, 3), np.uint8)
binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)

# Connected components
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
    binary, connectivity=8
)

for i in range(1, num_labels):  # skip background
    area = stats[i, cv2.CC_STAT_AREA]

    # Relaxed area bounds
    if area < 40 or area > 1500:
        continue

    cx, cy = centroids[i]
    cx, cy = int(cx), int(cy)

    # Ignore camps
    skip = False
    for camp in camps:
        if distance((cx, cy), camp["pos"]) < 30:
            skip = True
            break
    if skip:
        continue

    # Priority heuristic (area-based)
    if area > 700:
        priority = 6
    elif area > 300:
        priority = 4
    else:
        priority = 2

    casualties.append({
        "id": casualty_id,
        "pos": (cx, cy),
        "priority": priority
    })

    casualty_id += 1

print("Detected casualties:", casualties)



# camp priority and rescue ratio


camp_priority = {}

for camp_name, cas_ids in assignments.items():
    camp_priority[camp_name] = sum(
        c["priority"] for c in casualties if c["id"] in cas_ids
    )

print("Camp priorities:", camp_priority)

total_priority = sum(camp_priority.values())
total_casualties = len(casualties)

Pr = total_priority / total_casualties if total_casualties else 0
print("Rescue Ratio (Pr):", Pr)


Detected camps: [{'name': 'blue', 'pos': (321, 269), 'capacity': 4}, {'name': 'pink', 'pos': (80, 94), 'capacity': 3}, {'name': 'grey', 'pos': (367, 215), 'capacity': 2}]
Detected casualties: [{'id': 0, 'pos': (147, 31), 'priority': 2}, {'id': 1, 'pos': (192, 48), 'priority': 4}, {'id': 2, 'pos': (167, 74), 'priority': 2}, {'id': 3, 'pos': (110, 96), 'priority': 2}, {'id': 4, 'pos': (183, 114), 'priority': 2}, {'id': 5, 'pos': (100, 118), 'priority': 2}, {'id': 6, 'pos': (346, 128), 'priority': 2}, {'id': 7, 'pos': (332, 123), 'priority': 2}, {'id': 8, 'pos': (183, 132), 'priority': 2}, {'id': 9, 'pos': (327, 136), 'priority': 2}, {'id': 10, 'pos': (338, 144), 'priority': 2}, {'id': 11, 'pos': (187, 154), 'priority': 2}, {'id': 12, 'pos': (189, 178), 'priority': 2}, {'id': 13, 'pos': (180, 206), 'priority': 2}, {'id': 14, 'pos': (172, 230), 'priority': 2}, {'id': 15, 'pos': (170, 249), 'priority': 2}, {'id': 16, 'pos': (262, 277), 'priority': 4}, {'id': 17, 'pos': (156, 274), 'priority

In [ ]:
# TRIAGE SYSTEM 


import cv2
import numpy as np
import math


img = cv2.imread("./task_images/1.png")
assert img is not None, "Image not found!"

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)


# distance function


def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)


# clustering function 


def cluster_points(points, dist_thresh=30):
    clusters = []

    for p in points:
        added = False
        for cluster in clusters:
            if distance(p, cluster[0]) < dist_thresh:
                cluster.append(p)
                added = True
                break
        if not added:
            clusters.append([p])

    return clusters


# camp detection ( HSV color fragmentation)


camps = []

blue_mask = cv2.inRange(hsv, np.array([95, 100, 100]), np.array([135, 255, 255]))
pink_mask = cv2.inRange(hsv, np.array([140, 50, 50]), np.array([170, 255, 255]))
grey_mask = cv2.inRange(hsv, np.array([0, 0, 180]), np.array([180, 40, 255]))

camp_masks = {
    "blue": (blue_mask, 4),
    "pink": (pink_mask, 3),
    "grey": (grey_mask, 2)
}

for name, (mask, capacity) in camp_masks.items():
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    largest = None
    max_area = 0

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > max_area:
            max_area = area
            largest = cnt

    if largest is None:
        continue

    M = cv2.moments(largest)
    if M["m00"] == 0:
        continue

    cx = int(M["m10"] / M["m00"])
    cy = int(M["m01"] / M["m00"])

    camps.append({
        "name": name,
        "pos": (cx, cy),
        "capacity": capacity
    })

print("Detected camps:", camps)


# causualties detection

binary = cv2.adaptiveThreshold(
    gray,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV,
    11,
    2
)

kernel = np.ones((3, 3), np.uint8)
binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)

num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)

raw_points = []

for i in range(1, num_labels):  # skip background
    area = stats[i, cv2.CC_STAT_AREA]

    if area < 40 or area > 1500:
        continue

    cx, cy = centroids[i]
    cx, cy = int(cx), int(cy)

    # Ignore camp regions
    skip = False
    for camp in camps:
        if distance((cx, cy), camp["pos"]) < 30:
            skip = True
            break
    if skip:
        continue

    raw_points.append((cx, cy))

# clustering fragments into single casualties

clusters = cluster_points(raw_points, dist_thresh=30)

casualties = []
casualty_id = 0

for cluster in clusters:
    if len(cluster) < 2:
        continue  # ignore noise

    cx = sum(p[0] for p in cluster) // len(cluster)
    cy = sum(p[1] for p in cluster) // len(cluster)

    # Priority based on cluster size
    if len(cluster) >= 6:
        priority = 6
    elif len(cluster) >= 4:
        priority = 4
    else:
        priority = 2

    casualties.append({
        "id": casualty_id,
        "pos": (cx, cy),
        "priority": priority
    })

    casualty_id += 1

print("Detected casualties:", casualties)


# main triage assessments


assignments = {c["name"]: [] for c in camps}

for casualty in sorted(casualties, key=lambda x: -x["priority"]):
    best_camp = None
    best_dist = float("inf")

    for camp in camps:
        cname = camp["name"]

        if len(assignments[cname]) >= camp["capacity"]:
            continue

        d = distance(casualty["pos"], camp["pos"])
        if d < best_dist:
            best_dist = d
            best_camp = cname

    if best_camp:
        assignments[best_camp].append(casualty["id"])


# camp priority and rescue ratio


camp_priority = {}

for camp_name, ids in assignments.items():
    camp_priority[camp_name] = sum(
        c["priority"] for c in casualties if c["id"] in ids
    )

print("Camp priorities:", camp_priority)

total_priority = sum(camp_priority.values())
total_casualties = len(casualties)

Pr = total_priority / total_casualties if total_casualties else 0
print("Rescue Ratio (Pr):", Pr)


Detected camps: [{'name': 'blue', 'pos': (324, 273), 'capacity': 4}, {'name': 'pink', 'pos': (80, 94), 'capacity': 3}, {'name': 'grey', 'pos': (367, 215), 'capacity': 2}]
Detected casualties: [{'id': 0, 'pos': (105, 107), 'priority': 2}, {'id': 1, 'pos': (183, 123), 'priority': 2}, {'id': 2, 'pos': (335, 132), 'priority': 4}, {'id': 3, 'pos': (188, 166), 'priority': 2}, {'id': 4, 'pos': (176, 218), 'priority': 2}, {'id': 5, 'pos': (163, 261), 'priority': 2}, {'id': 6, 'pos': (74, 313), 'priority': 2}, {'id': 7, 'pos': (98, 357), 'priority': 2}, {'id': 8, 'pos': (44, 362), 'priority': 2}, {'id': 9, 'pos': (81, 418), 'priority': 2}, {'id': 10, 'pos': (463, 422), 'priority': 2}, {'id': 11, 'pos': (68, 458), 'priority': 2}, {'id': 12, 'pos': (102, 485), 'priority': 4}]
Camp priorities: {'blue': 10, 'pink': 6, 'grey': 6}
Rescue Ratio (Pr): 1.6923076923076923


In [ ]:
import cv2
import numpy as np
import math


img = cv2.imread("./task_images/1.png")
if img is None:
    # Fallback if the above fails
    img = cv2.imread("1.jpg") 

assert img is not None, "Image not found Check the file path."

hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)


# camp detection ( HSV Color segmentation)

camps = []

# Masks for the large circular camps
camp_masks = {
    "blue":  (cv2.inRange(hsv, np.array([95, 100, 100]), np.array([135, 255, 255])), 4),
    "pink":  (cv2.inRange(hsv, np.array([140, 50, 50]),  np.array([170, 255, 255])), 3),
    "grey":  (cv2.inRange(hsv, np.array([0, 0, 180]),    np.array([180, 40, 255])),  2)
}

for name, (mask, capacity) in camp_masks.items():
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        M = cv2.moments(largest)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            camps.append({"name": name, "pos": (cx, cy), "capacity": capacity})

print("Detected camps:", camps)


# casualty detection ( using color and SHAPE)

# This mask excludes the background (Dark Blue/Green) to find the small icons
# We are looking for Yellow, Green, and Orange/Red shapes


lower_cas = np.array([0, 70, 70])
upper_cas = np.array([90, 255, 255])
cas_mask = cv2.inRange(hsv, lower_cas, upper_cas)

# Cleaning up noise
kernel = np.ones((3,3), np.uint8)
cas_mask = cv2.morphologyEx(cas_mask, cv2.MORPH_OPEN, kernel)

contours, _ = cv2.findContours(cas_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

casualties = []
cas_id = 0

for cnt in contours:
    area = cv2.contourArea(cnt)
    # Filtering out background noise and the large camps
    if area < 50 or area > 1000:
        continue

    # Getting Center
    M = cv2.moments(cnt)
    if M["m00"] == 0: continue
    cx = int(M["m10"] / M["m00"])
    cy = int(M["m01"] / M["m00"])

    # shape classifications
    peri = cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, 0.04 * peri, True)
    vertices = len(approx)

    if vertices == 3:
        priority = 2    # Triangle
    elif vertices == 4:
        priority = 4    # Square
    else:
        priority = 6    # Star (usually 5-10 vertices depending on noise)

    casualties.append({
        "id": cas_id,
        "pos": (cx, cy),
        "priority": priority
    })
    cas_id += 1

print(f"Detected {len(casualties)} casualties:", casualties)


# main triage assessment

assignments = {c["name"]: [] for c in camps}

# Sort casualties by highest priority first
for casualty in sorted(casualties, key=lambda x: -x["priority"]):
    best_camp = None
    best_dist = float("inf")

    for camp in camps:
        cname = camp["name"]
        if len(assignments[cname]) < camp["capacity"]:
            d = distance(casualty["pos"], camp["pos"])
            if d < best_dist:
                best_dist = d
                best_camp = cname

    if best_camp:
        assignments[best_camp].append(casualty["id"])

# Final metrics


camp_priority = {}
for camp_name, ids in assignments.items():
    camp_priority[camp_name] = sum(
        c["priority"] for c in casualties if c["id"] in ids
    )

total_priority = sum(camp_priority.values())
total_casualties = len(casualties)
Pr = total_priority / total_casualties if total_casualties else 0

print("--- RESULTS ---")
print("Camp priority scores:", camp_priority)
print(f"Final Rescue Ratio (Pr): {Pr:.4f}")

Detected camps: [{'name': 'blue', 'pos': (310, 262), 'capacity': 4}, {'name': 'pink', 'pos': (109, 151), 'capacity': 3}, {'name': 'grey', 'pos': (450, 460), 'capacity': 2}]
Detected 9 casualties: [{'id': 0, 'pos': (330, 453), 'priority': 6}, {'id': 1, 'pos': (272, 396), 'priority': 4}, {'id': 2, 'pos': (121, 384), 'priority': 4}, {'id': 3, 'pos': (361, 379), 'priority': 4}, {'id': 4, 'pos': (460, 358), 'priority': 2}, {'id': 5, 'pos': (214, 259), 'priority': 2}, {'id': 6, 'pos': (228, 166), 'priority': 6}, {'id': 7, 'pos': (420, 77), 'priority': 4}, {'id': 8, 'pos': (185, 65), 'priority': 6}]
--- RESULTS ---
Camp priority scores: {'blue': 14, 'pink': 14, 'grey': 10}
Final Rescue Ratio (Pr): 4.2222
